# Tập 2.3 · Hiệu chỉnh camera bằng bàn cờ ♟️

[![Mở trên Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhpro82/tu-hoc-robot-tu-hanh/blob/main/tap23/tap23_hieu_chinh.ipynb)

Series **Tự Học Robot Tự Hành** · Mùa 2 · ⏱ ~50 phút · Giáo trình: Principles of Robot Autonomy, [Chương 6](https://porabook.com/book/camera-models/) (Mục 6.5.2)

Chụp 10 tấm bàn cờ để tìm lại 11 tham số của camera (phương pháp Zhang). Trong notebook này bạn:

1. Chạy lại đúng mô phỏng trong video.
2. **Tự viết** méo xuyên tâm, homography (DLT) và véc-tơ $v_{ij}$ của Zhang (3 bài), có ô chấm tự động.
3. Vẽ lưới méo và sai số chiếu lại.

Chạy lần lượt từng ô từ trên xuống (Shift + Enter).

## 0. Chuẩn bị (chạy 1 lần)

In [ ]:
import os, sys
if "google.colab" in sys.modules:  # trên Colab: tải code về
    if not os.path.exists("/content/tu-hoc-robot-tu-hanh"):
        !git clone -q https://github.com/thanhpro82/tu-hoc-robot-tu-hanh.git /content/tu-hoc-robot-tu-hanh
    os.chdir("/content/tu-hoc-robot-tu-hanh/tap23")
sys.path[:0] = ["bai_tap", "sim", ".."]

import numpy as np
import hieu_chinh as goc           # mô phỏng trong video (chụp 10 ảnh bàn cờ lúc import)
from ve_do_thi import ve           # vẽ hình
from IPython.display import Image, display
from kiem_tra import cham          # bộ chấm
print("Sẵn sàng!")

## 1. Chạy lại số liệu trong video

In [ ]:
print(f"Méo thật k = {goc.K_MEO:.1e}: góc ảnh (640,480) bị kéo lệch {goc.LECH_GOC_ANH:.1f} px")
print(f"Đoán điểm cách 2 m: dùng 500/320/240 bỏ méo lệch {goc.LECH_TRUOC * 100:.1f} cm · sau hiệu chỉnh lệch {goc.LECH_SAU * 1000:.1f} mm")
for h in goc.LICH_SU:
    K = h["K"]
    print(f"{h['buoc']}: k = {h['k']:.2e}  α={K[0,0]:.1f} β={K[1,1]:.1f} γ={K[0,1]:.2f} u0={K[0,2]:.1f} v0={K[1,2]:.1f}  RMS = {h['rms']:.2f} px")

## 2. Bài tập

| Bài | Hàm | Việc |
|---|---|---|
| **1** | `meo(uv, k, c)` | $d = uv - c$, $r^2 = \lvert d\rvert^2$, điểm méo $= (1 + k r^2)\,d + c$ |
| **2** | `tim_H(XY, uv)` | DLT: chuẩn hoá → ma trận A (2 dòng mỗi điểm) → SVD → bỏ chuẩn hoá → chia $H_{22}$ |
| **3** | `v_ij(H, i, j)` | 6 số theo công thức trong docstring (Eq. 6.28–6.31) |

Thay từng dòng `raise NotImplementedError(...)` bằng code của bạn, chạy ô đó, rồi chạy ô chấm ngay bên dưới (✅ đạt · ❌ sai, kèm gợi ý · ⏳ chưa làm).

In [ ]:
import numpy as np


def meo(uv, k, c):
    """Bài 1: méo xuyên tâm (Eq. 6.4). uv: (N, 2), k: hệ số méo, c: tâm méo (u_c, v_c).

    Với mỗi điểm: d = uv − c,  r² = d_u² + d_v²,  điểm bị méo = (1 + k·r²)·d + c.
    k < 0: méo thùng (điểm xa tâm bị kéo vào trong). Trả về mảng (N, 2)."""
    raise NotImplementedError("TODO Bài 1: méo xuyên tâm")


def chuan_hoa(P):
    """Có sẵn: ma trận 3×3 dời tâm đám điểm về gốc và co giãn cho khoảng cách trung bình = √2 (giúp DLT ổn định)."""
    m = P.mean(axis=0)
    s = np.sqrt(2) / np.linalg.norm(P - m, axis=1).mean()
    return np.array([[s, 0, -s * m[0]], [0, s, -s * m[1]], [0, 0, 1]])


def tim_H(XY, uv):
    """Bài 2: homography H (3×3) đưa điểm bàn cờ (X, Y, 1) thành điểm ảnh (u, v, 1) theo kiểu DLT.

    1. Chuẩn hoá:  T1 = chuan_hoa(XY), T2 = chuan_hoa(uv); a = T1·[X, Y, 1]ᵀ, b = T2·[u, v, 1]ᵀ cho từng điểm.
    2. Mỗi cặp (a = (X, Y, 1), b = (u, v, 1)) cho 2 dòng của ma trận A (9 cột):
         [−X, −Y, −1,  0,  0,  0, u·X, u·Y, u]
         [ 0,  0,  0, −X, −Y, −1, v·X, v·Y, v]
    3. h = véc-tơ kỳ dị ứng với giá trị kỳ dị nhỏ nhất: np.linalg.svd(A)[2][-1]; H_n = h.reshape(3, 3).
    4. Bỏ chuẩn hoá: H = inv(T2) @ H_n @ T1, rồi chia cho H[2, 2]."""
    raise NotImplementedError("TODO Bài 2: DLT tìm H")


def v_ij(H, i, j):
    """Bài 3: véc-tơ 6 số dùng để lập phương trình cho B = K⁻ᵀK⁻¹ (Eq. 6.28–6.31).

    Gọi h_i = cột thứ i của H (i = 0 hoặc 1), tức h_i = H[:, i]. Trả về np.array gồm 6 số:
        [h_i0·h_j0,  h_i0·h_j1 + h_i1·h_j0,  h_i1·h_j1,  h_i2·h_j0 + h_i0·h_j2,  h_i2·h_j1 + h_i1·h_j2,  h_i2·h_j2]
    Mỗi ảnh cho 2 phương trình: v_01·b = 0 và (v_00 − v_11)·b = 0 (vì r1 ⟂ r2 và |r1| = |r2|)."""
    raise NotImplementedError("TODO Bài 3: véc-tơ v_ij")


In [ ]:
cham(globals())

## 3. Xem kết quả

In [ ]:
import loi_giai
try:
    ve(globals(), "ket_qua.png")
except NotImplementedError:
    print("Chưa xong Bài 1, vẽ bằng lời giải.")
    ve(loi_giai, "ket_qua.png")
display(Image("ket_qua.png"))

## 4. Thử thách

Zhang cần bao nhiêu ảnh? Chạy với 3, 5, 10 ảnh đầu và so K với giá trị thật (512, 508, 331, 236).

In [ ]:
import loi_giai
from kiem_tra import DS_UV
for n in [3, 5, 10]:
    K, _, _ = goc.zhang(DS_UV[:n])
    print(f"{n:2d} ảnh: α={K[0,0]:6.1f} β={K[1,1]:6.1f} u0={K[0,2]:6.1f} v0={K[1,2]:6.1f}  (chưa tính méo)")

## Câu hỏi tự kiểm tra

Tự trả lời trước, rồi mở đáp án ở mục *Câu hỏi tự kiểm tra* trong [README của tập](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh/tree/main/tap23).

Code cả series: [github.com/thanhpro82/tu-hoc-robot-tu-hanh](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh)